# Backbone Inference — Aniemore/wavlm-emotion-russian-resd

Runs the pre-trained Aniemore model directly on RESD test set, computes metrics and confusion matrix.

## 1. Imports

In [ ]:
import warnings, sys, os
warnings.filterwarnings('ignore')

import numpy as np
import torch
import torchaudio.functional as AF
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm
from datasets import load_dataset
from transformers import AutoFeatureExtractor, AutoModelForAudioClassification
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score,
    f1_score, classification_report, confusion_matrix,
)

# add repo root to path if running from notebooks/
repo_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
if repo_root not in sys.path:
    sys.path.insert(0, repo_root)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)

## 2. Load model

In [ ]:
MODEL_NAME = 'Aniemore/wavlm-emotion-russian-resd'

processor = AutoFeatureExtractor.from_pretrained('microsoft/wavlm-base')
model = AutoModelForAudioClassification.from_pretrained(MODEL_NAME)
model.eval().to(device)

print('Model labels:', model.config.id2label)

## 3. Load RESD test set

In [ ]:
SAMPLE_RATE = 16_000
MAX_LEN = SAMPLE_RATE * 10  # 10 seconds

ds = load_dataset('Aniemore/resd')
test_ds = ds['test']
print(f'Test samples: {len(test_ds)}')

def decode_audio(speech_field):
    try:
        samples = speech_field.get_all_samples()
        waveform = samples.data
        if waveform.ndim > 1:
            waveform = waveform.mean(dim=0)
        return waveform.float(), int(samples.sample_rate)
    except AttributeError:
        return torch.tensor(speech_field['array'], dtype=torch.float32), speech_field['sampling_rate']

## 4. Build label mapping

Align model's label order with RESD ground-truth labels.

In [ ]:
# model.config.id2label: {model_id -> emotion_name}
# We build: model_id -> resd_id using the model's own label names
RESD_LABEL2ID = {
    'happiness': 0, 'sadness': 1, 'anger': 2, 'fear': 3,
    'disgust': 4, 'enthusiasm': 5, 'neutral': 6,
}
RESD_ID2LABEL = {v: k for k, v in RESD_LABEL2ID.items()}

# map model output index → RESD label id
model_id2resd = {}
for model_id, label_name in model.config.id2label.items():
    label_name_lower = label_name.lower()
    if label_name_lower in RESD_LABEL2ID:
        model_id2resd[int(model_id)] = RESD_LABEL2ID[label_name_lower]
    else:
        print(f'Warning: model label "{label_name}" not found in RESD labels')

print('Mapping (model_id → resd_id → name):')
for mid, rid in sorted(model_id2resd.items()):
    print(f'  {mid} → {rid} ({RESD_ID2LABEL[rid]})')

## 5. Run inference on test set

In [ ]:
all_preds, all_labels = [], []

with torch.no_grad():
    for ex in tqdm(test_ds, desc='Inference'):
        waveform, sr = decode_audio(ex['speech'])
        if sr != SAMPLE_RATE:
            waveform = AF.resample(waveform.unsqueeze(0), sr, SAMPLE_RATE).squeeze(0)
        waveform = waveform[:MAX_LEN]

        inputs = processor(waveform.numpy(), sampling_rate=SAMPLE_RATE, return_tensors='pt')
        inputs = {k: v.to(device) for k, v in inputs.items()}

        logits = model(**inputs).logits
        model_pred = logits.argmax(dim=-1).item()

        pred_resd = model_id2resd.get(model_pred, model_pred)
        true_resd = RESD_LABEL2ID[ex['emotion']]

        all_preds.append(pred_resd)
        all_labels.append(true_resd)

all_preds  = np.array(all_preds)
all_labels = np.array(all_labels)
print('Done.')

## 6. Metrics

In [ ]:
label_names = [RESD_ID2LABEL[i] for i in range(7)]

print(f'accuracy          {accuracy_score(all_labels, all_preds):.4f}')
print(f'weighted_accuracy {balanced_accuracy_score(all_labels, all_preds):.4f}')
print(f'f1_macro          {f1_score(all_labels, all_preds, average="macro", zero_division=0):.4f}')
print(f'f1_weighted       {f1_score(all_labels, all_preds, average="weighted", zero_division=0):.4f}')
print()
print(classification_report(all_labels, all_preds, target_names=label_names, zero_division=0))

## 7. Confusion matrix

In [ ]:
cm = confusion_matrix(all_labels, all_preds)

fig, ax = plt.subplots(figsize=(9, 7))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=label_names, yticklabels=label_names, ax=ax)
ax.set_xlabel('Predicted')
ax.set_ylabel('True')
ax.set_title(f'Confusion Matrix — {MODEL_NAME}')
plt.tight_layout()
plt.show()